In [35]:
import json
import pandas as pd

In [36]:
def load_data(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

In [37]:
train_file = "train_short.json"
train = load_data(train_file)

In [38]:
# convert json to dataframe
train_df = pd.DataFrame(train)

In [39]:
# convert json to dataframe
train_df = pd.DataFrame(train)

# traindf where wellformedanswers is not empty
train_df.drop(columns=["query_id", "query_type", "wellFormedAnswers"], inplace=True)

# for answer in column answers apply lambda function to get the first element
train_df["answers"] = train_df["answers"].apply(lambda x: x[0] if len(x) > 0 else None)

# split passages into twenty columns (passage_0_text, passage_0_is_selected, passage_1_text, passage_1_is_selected, ...)
for i in range(10):
    passage_text = train_df["passages"].apply(lambda x: x[i]["passage_text"])
    passage_is_selected = train_df["passages"].apply(lambda x: x[i]["is_selected"])
    train_df[f"passage_{i}_selected"] = passage_is_selected
    train_df[f"passage_{i}_text"] = passage_text

# drop columns passage
train_df.drop(columns=["passages"], inplace=True)

train_df

,answers,query,passage_0_selected,passage_0_text,passage_1_selected,passage_1_text,passage_2_selected,passage_2_text,passage_3_selected,passage_3_text,...,passage_5_selected,passage_5_text,passage_6_selected,passage_6_text,passage_7_selected,passage_7_text,passage_8_selected,passage_8_text,passage_9_selected,passage_9_text
0,The immediate impact of the success of the man...,)what was the immediate impact of the success ...,1,The presence of communication amid scientific ...,0,The Manhattan Project and its atomic bomb help...,0,Essay on The Manhattan Project - The Manhattan...,0,The Manhattan Project was the name for a proje...,...,0,The Manhattan Project. This once classified ph...,0,Nor will it attempt to substitute for the extr...,0,Manhattan Project. The Manhattan Project was a...,0,"In June 1942, the United States Army Corps of ...",0,One of the main reasons Hanford was selected a...
1,Restorative justice that fosters dialogue betw...,_________ justice is designed to repair the ha...,0,"group discussions, community boards or panels ...",0,punishment designed to repair the damage done ...,0,Tutorial: Introduction to Restorative Justice....,0,"Organize volunteer community panels, boards, o...",...,0,Each of these types of communities—the geograp...,1,The approach is based on a theory of justice t...,0,Inherent in many people’s understanding of the...,0,"Criminal justice, however, is not usually conc...",0,The circle includes a wide range of participan...
2,The customer care number of Amex India is 1800...,amex india customer care number,0,American Express is the world's premier servic...,0,Customer Service Main Page: Get Information: U...,1,Amex Card India Customer Care Phone Number Pho...,0,Air India American Express Gold Card Customer ...,...,0,"Update Address, Payee Name, Bank Account and C...",0,Please mention the first 11 digits of your Ame...,0,"The postal and official address, email address...",0,Corporate Cardmembers can contact the 24 hour ...,0,Through which any customers can easily contact...
3,"Ramen is a quick-cooking noodles, typically se...",definition of ramen,0,"Ramen is of Chinese origin, however it is uncl...",0,"ramen definition, meaning, what is ramen: a Ja...",0,Definition of ramen written for English Langua...,0,Sapporo ramen comes from Japan's northernmost ...,...,0,Wiktionary (0.00 / 0 votes) Rate this definiti...,0,Related dishes. 1 Nagasaki champon. The noodl...,0,[sometimes with sing. v.] Japanese noodles of ...,0,"‘The noodles, most of which we left behind bec...",0,"On the ground floor level, there is a souvenir..."
4,Rachel Carson died because of cancer.,why did rachel carson die,0,A Fish and Wildlife Service National Wildlife ...,0,Though much of their correspondence was destro...,0,The impetus for Silent Spring was a letter wri...,0,How about the environmentalist and writer Rach...,...,0,"Rachel Carson was born on May 27, 1907, on a f...",0,Lived 1907 – 1964. Rachel Carson played a key ...,0,Rachel Carson had a large love for nature and ...,1,Heroes commit feats of courage involving risk ...,0,"To passers-by the mother would say, pointing, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,The Eastern part of Texas is in the Central Ti...,is texas central time or eastern time,0,Central Time Zone: Ward County: Central Time Z...,0,What time zone is austin texas? eastern? pacif...,0,Western half of the state: Central Standard Ti...,1,Texas Time Zone . The Eastern part of Texas (m...,...,0,Regions in the Central Time Zone prior to 2015...,0,Manitoba is the only Canadian province that ob...,0,CDT is observed in the following United States...,0,Texas Time ↔ Greenwich Mean Time; Texas Time ↔...,0,1.The Eastern time zone and the Central time z...
9996,No Answer Present.,is the acl located inside the synovial membrane,0,Notice: RSNA will be launching a new online ed...,0,This form of synovitis is a more serious condi...,0,The cruciate ligaments are so cal

In [40]:
sample_queries = list(train_df['query'])
expected_responses = list(train_df['answers'])

# create a list of every passage, also append the rowid stretched to 4 digits + passage_0_selected number
passages = []
for i, row in train_df.iterrows():
    for j in range(10):
        passage = row[f"passage_{j}_text"]
        passage_is_selected = row[f"passage_{j}_selected"]
        id = f"{int(i):04d}_{passage_is_selected}"
        passages.append((id, passage))

---
---

In [41]:
import os
from langchain_core.documents import Document

content_list = [
    "Andrew Ng is the CEO of Landing AI and is known for his pioneering work in deep learning. He is also widely recognized for democratizing AI education through platforms like Coursera.",
    "Sam Altman is the CEO of OpenAI and has played a key role in advancing AI research and development. He is a strong advocate for creating safe and beneficial AI technologies.",
    "Demis Hassabis is the CEO of DeepMind and is celebrated for his innovative approach to artificial intelligence. He gained prominence for developing systems that can master complex games like AlphaGo.",
    "Sundar Pichai is the CEO of Google and Alphabet Inc., and he is praised for leading innovation across Google's vast product ecosystem. His leadership has significantly enhanced user experiences on a global scale.",
    "Arvind Krishna is the CEO of IBM and is recognized for transforming the company towards cloud computing and AI solutions. He focuses on providing cutting-edge technologies to address modern business challenges.",
]

langchain_documents = []

for content in content_list:
    langchain_documents.append(
        Document(
            page_content=content,
        )
    )

In [42]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="mxbai-embed-large")
vector_store = InMemoryVectorStore(embeddings)

vector_store.add_documents(langchain_documents)

['064734f1-5c52-4f5a-bb36-ef5711f58d05',
 'e8816233-a3be-4580-81e6-480571274be0',
 '3efd854e-a87c-489f-9eb2-53dbe9de40da',
 '670422cc-e0f8-452b-8fcc-374f6220f291',
 '9f960f68-0633-4947-8135-598b4945db04']

In [43]:
retriever = vector_store.as_retriever(search_kwargs={"k": 1})

In [44]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="gemma3:4b", temperature=0.0, max_tokens=512)

In [45]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


template = """Answer the question based only on the following context:
{context}

Question: {query}
"""
prompt = ChatPromptTemplate.from_template(template)

qa_chain = prompt | llm | StrOutputParser()

In [46]:
def format_docs(relevant_docs):
    return "\n".join(doc.page_content for doc in relevant_docs)


query = "Who is the CEO of OpenAI?"

relevant_docs = retriever.invoke(query)
qa_chain.invoke({"context": format_docs(relevant_docs), "query": query})

'Sam Altman is the CEO of OpenAI.'

In [47]:
sample_queries = [
    "Which CEO is widely recognized for democratizing AI education through platforms like Coursera?",
    "Who is Sam Altman?",
    "Who is Demis Hassabis and how did he gained prominence?",
    "Who is the CEO of Google and Alphabet Inc., praised for leading innovation across Google's product ecosystem?",
    "How did Arvind Krishna transformed IBM?",
]

expected_responses = [
    "Andrew Ng is the CEO of Landing AI and is widely recognized for democratizing AI education through platforms like Coursera.",
    "Sam Altman is the CEO of OpenAI and has played a key role in advancing AI research and development. He strongly advocates for creating safe and beneficial AI technologies.",
    "Demis Hassabis is the CEO of DeepMind and is celebrated for his innovative approach to artificial intelligence. He gained prominence for developing systems like AlphaGo that can master complex games.",
    "Sundar Pichai is the CEO of Google and Alphabet Inc., praised for leading innovation across Google's vast product ecosystem. His leadership has significantly enhanced user experiences globally.",
    "Arvind Krishna is the CEO of IBM and has transformed the company towards cloud computing and AI solutions. He focuses on delivering cutting-edge technologies to address modern business challenges.",
]

In [48]:
from ragas import EvaluationDataset


dataset = []

for query, reference in zip(sample_queries, expected_responses):
    relevant_docs = retriever.invoke(query)
    response = qa_chain.invoke({"context": format_docs(relevant_docs), "query": query})
    dataset.append(
        {
            "user_input": query,
            "retrieved_contexts": [rdoc.page_content for rdoc in relevant_docs],
            "response": response,
            "reference": reference,
        }
    )

evaluation_dataset = EvaluationDataset.from_list(dataset)

c:\Users\piotr\Desktop\cruise-screening\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [49]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness

evaluator_llm = LangchainLLMWrapper(llm)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
    llm=evaluator_llm,
)

result

Evaluating:  80%|████████  | 12/15 [02:52<00:51, 17.11s/it]Exception raised in Job[2]: TimeoutError()
Exception raised in Job[11]: TimeoutError()
Exception raised in Job[14]: TimeoutError()
Evaluating: 100%|██████████| 15/15 [03:00<00:00, 12.00s/it]


{'context_recall': 1.0000, 'faithfulness': 1.0000, 'factual_correctness(mode=f1)': 1.0000}

In [50]:
dataset

[{'user_input': 'Which CEO is widely recognized for democratizing AI education through platforms like Coursera?',
  'retrieved_contexts': ['Andrew Ng is the CEO of Landing AI and is known for his pioneering work in deep learning. He is also widely recognized for democratizing AI education through platforms like Coursera.'],
  'response': 'Andrew Ng is widely recognized for democratizing AI education through platforms like Coursera.',
  'reference': 'Andrew Ng is the CEO of Landing AI and is widely recognized for democratizing AI education through platforms like Coursera.'},
 {'user_input': 'Who is Sam Altman?',
  'retrieved_contexts': ['Sam Altman is the CEO of OpenAI and has played a key role in advancing AI research and development. He is a strong advocate for creating safe and beneficial AI technologies.'],
  'response': 'Sam Altman is the CEO of OpenAI and has played a key role in advancing AI research and development. He is a strong advocate for creating safe and beneficial AI tec